In [3]:
import pandas as pd
import numpy as np
import ast
from scipy.sparse import load_npz


from sklearn.metrics.pairwise import cosine_similarity

In [4]:
df = pd.read_csv("../data/arxiv_cleaned.csv")

df.shape

(41127, 6)

In [5]:

tfidf_matrix = load_npz(
    "../models/tfidf_matrix.npz"
)

In [6]:
embiddings = np.load(
    "../models/transformer_embeddings.npy"
)

In [7]:
embiddings.shape

(41127, 384)

In [8]:
title_duplicates = df["titles"].duplicated().sum()

print("Duplicate titles:", title_duplicates)

Duplicate titles: 22


In [9]:
duplicate_titles = df[
    df["titles"].duplicated(keep=False)
].sort_values("titles")

duplicate_titles[["titles", "terms"]].head(20)

,titles,terms
13212,Class-Agnostic Segmentation Loss and Its Appli...,"['cs.CV', 'cs.LG']"
17097,Class-Agnostic Segmentation Loss and Its Appli...,"['cs.CV', 'cs.LG']"
24239,Deep Image Compositing,['cs.CV']
33182,Deep Image Compositing,['cs.CV']
1651,Deep Reinforcement Learning: An Overview,['cs.LG']
1748,Deep Reinforcement Learning: An Overview,"['cs.LG', 'cs.AI', 'stat.ML']"
3328,Distributed Online Learning with Multiple Kernels,['cs.LG']
3529,Distributed Online Learning with Multiple Kernels,"['cs.LG', 'stat.ML']"
19166,Dynamic Belief Fusion for Object Detection,['cs.CV']
19167,Dynamic Belief Fusion for Object Detection,['cs.CV']


In [10]:
id_duplicates = df.duplicated(
    subset=["titles", "abstracts"],
    keep=False
).sum()

print("Duplicate title + abstract:", id_duplicates)

Duplicate title + abstract: 14


In [11]:

duplicate_papers = df[
    df.duplicated(
        subset=["titles", "abstracts"],
        keep=False
    )
].sort_values("titles")

duplicate_papers[
    ["titles", "terms"]
]

,titles,terms
24900,Kernel-Guided Training of Implicit Generative ...,"['cs.LG', 'stat.ML']"
24903,Kernel-Guided Training of Implicit Generative ...,"['stat.ML', 'cs.LG', 'math.DS']"
67,Learnable Hypergraph Laplacian for Hypergraph ...,['cs.LG']
71,Learnable Hypergraph Laplacian for Hypergraph ...,"['cs.LG', 'cs.AI']"
4533,On Improving Deep Reinforcement Learning for P...,['cs.LG']
4551,On Improving Deep Reinforcement Learning for P...,"['cs.LG', 'stat.ML']"
11124,Probabilistic inverse reinforcement learning i...,"['cs.LG', 'stat.ML']"
11156,Probabilistic inverse reinforcement learning i...,"['stat.ML', 'cs.LG']"
38098,Scalable Matrix-valued Kernel Learning for Hig...,"['cs.LG', 'stat.ML']"
38193,Scalable Matrix-valued Kernel Learning for Hig...,"['stat.ML', 'cs.LG']"


In [12]:
df_eval = df.drop_duplicates(
    subset=["titles", "abstracts"]
).reset_index(drop=True)

print("Original rows:", len(df))
print("After deduplication:", len(df_eval))
print("Removed:", len(df) - len(df_eval))

Original rows: 41127
After deduplication: 41120
Removed: 7


In [13]:
original_indices = df.drop_duplicates(
    subset=["titles", "abstracts"]
).index

In [14]:
tfidf_matrix_eval = tfidf_matrix[original_indices]

embiddings_eval = embiddings[original_indices]

In [16]:
print("TF-IDF:", tfidf_matrix_eval.shape)
print("Transformer:", embiddings_eval.shape)
print("Data:", df_eval.shape)

TF-IDF: (41120, 50000)
Transformer: (41120, 384)
Data: (41120, 6)


In [17]:
import ast

def parse_terms(x):
    try:
        return set(ast.literal_eval(x))
    except:
        return set()

In [18]:
df_eval["parsed_terms"] = df_eval["terms"].apply(parse_terms)

In [19]:
df_eval[["terms", "parsed_terms"]].head()

,terms,parsed_terms
0,['cs.LG'],{cs.LG}
1,"['cs.LG', 'cs.AI']","{cs.LG, cs.AI}"
2,"['cs.LG', 'cs.CR', 'stat.ML']","{stat.ML, cs.LG, cs.CR}"
3,"['cs.LG', 'cs.CR']","{cs.LG, cs.CR}"
4,['cs.LG'],{cs.LG}


In [20]:
def get_relevance_score(query_index, paper_index):
    query_terms = df_eval.loc[query_index, "parsed_terms"]
    paper_terms = df_eval.loc[paper_index, "parsed_terms"]

    return len(query_terms.intersection(paper_terms))

In [21]:
query_index = 0

relevance_scores = []

for paper_index in range(len(df_eval)):
    if paper_index == query_index:
        continue

    score = get_relevance_score(
        query_index,
        paper_index
    )

    relevance_scores.append(score)

pd.Series(relevance_scores).value_counts().sort_index()

0    17822
1    23297
Name: count, dtype: int64

In [22]:
term_counts = df_eval["parsed_terms"].apply(len)

term_counts.value_counts().sort_index()

parsed_terms
1     15543
2     13211
3      8472
4      2786
5       861
6       194
7        45
8         6
9         1
11        1
Name: count, dtype: int64

In [23]:
multi_term_indices = df_eval.index[
    df_eval["parsed_terms"].apply(len) >= 2
]

print("Number of candidate queries:", len(multi_term_indices))
print("First 10:", multi_term_indices[:10].tolist())

Number of candidate queries: 25577
First 10: [1, 2, 3, 5, 7, 9, 11, 13, 15, 16]


In [24]:
query_indices = np.random.RandomState(42).choice(
    multi_term_indices,
    size=100,
    replace=False
)

print("Number of evaluation queries:", len(query_indices))
print("First 10 query indices:", query_indices[:10])

Number of evaluation queries: 100
First 10 query indices: [11931   865 37010 19990 31129 11962 30464  6839 30702 26808]


In [25]:
def get_relevance_scores(query_index):
    query_terms = df_eval.loc[query_index, "parsed_terms"]

    relevance_scores = np.zeros(len(df_eval), dtype=int)

    for paper_index in range(len(df_eval)):
        if paper_index == query_index:
            continue

        paper_terms = df_eval.loc[paper_index, "parsed_terms"]

        relevance_scores[paper_index] = len(
            query_terms.intersection(paper_terms)
        )

    return relevance_scores

In [26]:
query_index = query_indices[0]

relevance_scores = get_relevance_scores(query_index)

print("Query index:", query_index)
print("Query terms:", df_eval.loc[query_index, "parsed_terms"])

print(
    pd.Series(relevance_scores)
    .value_counts()
    .sort_index()
)

Query index: 11931
Query terms: {'cs.LG', 'cs.CV'}
0      456
1    34566
2     6098
Name: count, dtype: int64


In [ ]:
query_index = query_indices[0]

relevance_scores = get_relevance_scores(query_index)

def get_tfidf_recommendations(query_index, top_k=10):
    scores = cosine_similarity(
        tfidf_matrix_eval[query_index],
        tfidf_matrix_eval
    ).ravel()

    scores[query_index] = -np.inf
    return np.argsort(scores)[::-1][:top_k]


def get_transformer_recommendations(query_index, top_k=10):
    scores = cosine_similarity(
        embiddings_eval[query_index].reshape(1, -1),
        embiddings_eval
    ).ravel()

    scores[query_index] = -np.inf
    return np.argsort(scores)[::-1][:top_k]


query_index = query_indices[0]
relevance_scores = get_relevance_scores(query_index)

tfidf_recommendations = get_tfidf_recommendations(query_index, top_k=10)
transformer_recommendations = get_transformer_recommendations(
    query_index,
    top_k=10
)

print("TF-IDF relevance:")
print(relevance_scores[tfidf_recommendations])

print("\nTransformer relevance:")
print(relevance_scores[transformer_recommendations])

transformer_recommendations = get_transformer_recommendations(
    query_index,
    top_k=10
)

print("TF-IDF relevance:")
print(relevance_scores[tfidf_recommendations])

print("\nTransformer relevance:")
print(relevance_scores[transformer_recommendations])

NameError: name 'get_tfidf_recommendations' is not defined

In [ ]:
from sklearn.metrics import ndcg_score

In [ ]:
def get_tfidf_scores(query_index):
    return cosine_similarity(
        tfidf_matrix_eval[query_index],
        tfidf_matrix_eval
    ).flatten()

In [ ]:
def get_transformer_scores(query_index):
    return cosine_similarity(
        embeddings_eval[query_index].reshape(1, -1),
        embeddings_eval
    ).flatten()

In [ ]:
from sklearn.metrics import ndcg_score

def ndcg_at_k(model_scores, relevance_scores, query_index, k=10):
    model_scores = model_scores.copy()

    # Exclude the query paper itself
    model_scores[query_index] = -np.inf

    return ndcg_score(
        [relevance_scores],
        [model_scores],
        k=k
    )

In [ ]:
def ndcg_at_k(model_scores, relevance_scores, query_index, k=10):
    model_scores = model_scores.copy()

    # استبعاد الـ query نفسها
    model_scores[query_index] = -np.inf

    # ترتيب الأوراق حسب similarity score
    ranked_indices = np.argsort(model_scores)[::-1]

    # أول K recommendations
    top_indices = ranked_indices[:k]

    # relevance الحقيقية لأول K أوراق
    top_relevance = relevance_scores[top_indices]

    # model scores لأول K أوراق
    top_scores = model_scores[top_indices]

    return ndcg_score(
        [top_relevance],
        [top_scores],
        k=k
    )

In [ ]:
query_index = query_indices[0]

relevance_scores = get_relevance_scores(query_index)

tfidf_scores = get_tfidf_scores(query_index)
transformer_scores = get_transformer_scores(query_index)

tfidf_ndcg = ndcg_at_k(
    tfidf_scores,
    relevance_scores,
    query_index,
    k=10
)

transformer_ndcg = ndcg_at_k(
    transformer_scores,
    relevance_scores,
    query_index,
    k=10
)

print("TF-IDF NDCG@10:", tfidf_ndcg)
print("Transformer NDCG@10:", transformer_ndcg)

TF-IDF NDCG@10: 0.8635485652165326
Transformer NDCG@10: 0.9036566975408976


In [ ]:
tfidf_ndcg_scores = []

for query_index in query_indices:
    relevance_scores = get_relevance_scores(query_index)
    model_scores = get_tfidf_scores(query_index)

    score = ndcg_at_k(
        model_scores,
        relevance_scores,
        query_index,
        k=10
    )

    tfidf_ndcg_scores.append(score)

print("Number of evaluated queries:", len(tfidf_ndcg_scores))
print("Average TF-IDF NDCG@10:", np.mean(tfidf_ndcg_scores))

Number of evaluated queries: 100
Average TF-IDF NDCG@10: 0.8859476486120179


In [ ]:
transformer_ndcg_scores = []

for query_index in query_indices:
    relevance_scores = get_relevance_scores(query_index)
    model_scores = get_transformer_scores(query_index)

    score = ndcg_at_k(
        model_scores,
        relevance_scores,
        query_index,
        k=10
    )

    transformer_ndcg_scores.append(score)

print("Number of evaluated queries:", len(transformer_ndcg_scores))
print(
    "Average Transformer NDCG@10:",
    np.mean(transformer_ndcg_scores)
)

Number of evaluated queries: 100
Average Transformer NDCG@10: 0.8996303144359064


In [ ]:
np.mean(transformer_ndcg_scores)

np.float64(0.8996303144359064)

In [ ]:
def precision_at_k(model_scores, relevance_scores, query_index, k=10):
    model_scores = model_scores.copy()

    # Exclude the query itself
    model_scores[query_index] = -np.inf

    ranked_indices = np.argsort(model_scores)[::-1]
    top_indices = ranked_indices[:k]

    relevant_retrieved = np.sum(
        relevance_scores[top_indices] > 0
    )

    return relevant_retrieved / k

In [ ]:
def recall_at_k(model_scores, relevance_scores, query_index, k=10):
    model_scores = model_scores.copy()

    # Exclude the query itself
    model_scores[query_index] = -np.inf

    ranked_indices = np.argsort(model_scores)[::-1]
    top_indices = ranked_indices[:k]

    relevant_retrieved = np.sum(
        relevance_scores[top_indices] > 0
    )

    total_relevant = np.sum(
        relevance_scores > 0
    )

    return relevant_retrieved / total_relevant

In [ ]:
tfidf_precision_scores = []
tfidf_recall_scores = []

for query_index in query_indices:

    relevance_scores = get_relevance_scores(query_index)
    model_scores = get_tfidf_scores(query_index)

    precision = precision_at_k(
        model_scores,
        relevance_scores,
        query_index,
        k=10
    )

    recall = recall_at_k(
        model_scores,
        relevance_scores,
        query_index,
        k=10
    )

    tfidf_precision_scores.append(precision)
    tfidf_recall_scores.append(recall)

print("Average TF-IDF Precision@10:",
      np.mean(tfidf_precision_scores))

print("Average TF-IDF Recall@10:",
      np.mean(tfidf_recall_scores))

Average TF-IDF Precision@10: 0.9060000000000002
Average TF-IDF Recall@10: 0.00033174113051003776


In [ ]:
transformer_precision_scores = []
transformer_recall_scores = []

for query_index in query_indices:

    relevance_scores = get_relevance_scores(query_index)
    model_scores = get_transformer_scores(query_index)

    precision = precision_at_k(
        model_scores,
        relevance_scores,
        query_index,
        k=10
    )

    recall = recall_at_k(
        model_scores,
        relevance_scores,
        query_index,
        k=10
    )

    transformer_precision_scores.append(precision)
    transformer_recall_scores.append(recall)

print("Average Transformer Precision@10:",
      np.mean(transformer_precision_scores))

print("Average Transformer Recall@10:",
      np.mean(transformer_recall_scores))

Average Transformer Precision@10: 0.9480000000000001
Average Transformer Recall@10: 0.00034880432669578943


In [ ]:
def mrr_at_k(model_scores, relevance_scores, query_index, k=10):
    model_scores = model_scores.copy()

    # Exclude the query itself
    model_scores[query_index] = -np.inf

    ranked_indices = np.argsort(model_scores)[::-1]
    top_indices = ranked_indices[:k]

    for rank, index in enumerate(top_indices, start=1):
        if relevance_scores[index] > 0:
            return 1 / rank

    return 0.0

In [ ]:
tfidf_mrr_scores = []

for query_index in query_indices:
    relevance_scores = get_relevance_scores(query_index)
    model_scores = get_tfidf_scores(query_index)

    score = mrr_at_k(
        model_scores,
        relevance_scores,
        query_index,
        k=10
    )

    tfidf_mrr_scores.append(score)

print(
    "Average TF-IDF MRR@10:",
    np.mean(tfidf_mrr_scores)
)

Average TF-IDF MRR@10: 0.9541666666666667


In [ ]:
transformer_mrr_scores = []

for query_index in query_indices:
    relevance_scores = get_relevance_scores(query_index)
    model_scores = get_transformer_scores(query_index)

    score = mrr_at_k(
        model_scores,
        relevance_scores,
        query_index,
        k=10
    )

    transformer_mrr_scores.append(score)

print(
    "Average Transformer MRR@10:",
    np.mean(transformer_mrr_scores)
)

Average Transformer MRR@10: 0.975


In [ ]:
def average_precision_at_k(
    model_scores,
    relevance_scores,
    query_index,
    k=10
):
    model_scores = model_scores.copy()

    # Exclude the query itself
    model_scores[query_index] = -np.inf

    ranked_indices = np.argsort(model_scores)[::-1]
    top_indices = ranked_indices[:k]

    relevant_count = 0
    precision_sum = 0.0

    for rank, index in enumerate(top_indices, start=1):

        if relevance_scores[index] > 0:
            relevant_count += 1
            precision_sum += relevant_count / rank

    if relevant_count == 0:
        return 0.0

    return precision_sum / relevant_count

In [ ]:
tfidf_ap_scores = []

for query_index in query_indices:

    relevance_scores = get_relevance_scores(query_index)
    model_scores = get_tfidf_scores(query_index)

    score = average_precision_at_k(
        model_scores,
        relevance_scores,
        query_index,
        k=10
    )

    tfidf_ap_scores.append(score)

print(
    "Average TF-IDF AP@10:",
    np.mean(tfidf_ap_scores)
)

Average TF-IDF AP@10: 0.9319926917989418


In [ ]:
transformer_ap_scores = []

for query_index in query_indices:

    relevance_scores = get_relevance_scores(query_index)
    model_scores = get_transformer_scores(query_index)

    score = average_precision_at_k(
        model_scores,
        relevance_scores,
        query_index,
        k=10
    )

    transformer_ap_scores.append(score)

print(
    "Average Transformer AP@10:",
    np.mean(transformer_ap_scores)
)

Average Transformer AP@10: 0.964053303728899


In [ ]:
evaluation_results = pd.DataFrame({
    "Metric": [
        "Precision@10",
        "Recall@10",
        "NDCG@10",
        "MRR@10",
        "AP@10"
    ],
    "TF-IDF": [
        np.mean(tfidf_precision_scores),
        np.mean(tfidf_recall_scores),
        np.mean(tfidf_ndcg_scores),
        np.mean(tfidf_mrr_scores),
        np.mean(tfidf_ap_scores)
    ],
    "Transformer": [
        np.mean(transformer_precision_scores),
        np.mean(transformer_recall_scores),
        np.mean(transformer_ndcg_scores),
        np.mean(transformer_mrr_scores),
        np.mean(transformer_ap_scores)
    ]
})

evaluation_results

NameError: name 'pd' is not defined

In [ ]:
evaluation_results.to_csv(
    "../data/evaluation_results.csv",
    index=False
)

print("Evaluation results saved successfully.")

NameError: name 'evaluation_results' is not defined

In [ ]:
#print("Query terms:")
#print(df_eval.loc[0, "parsed_terms"])

Query terms:
{'cs.LG'}


In [ ]:
#for idx in relevant_indices[:10]:
   # print(idx, df_eval.loc[idx, "parsed_terms"])

1 {'cs.AI', 'cs.LG'}
2 {'cs.LG', 'cs.CR', 'stat.ML'}
3 {'cs.LG', 'cs.CR'}
4 {'cs.LG'}
5 {'cs.LG', 'stat.ML'}
6 {'cs.LG'}
7 {'cs.LG', 'stat.ML'}
8 {'cs.LG'}
9 {'cs.AI', 'cs.LG'}
10 {'cs.LG'}
